<!-- @format -->

# 04 - Advanced Models & Hyperparameter Tuning

Notebook này thực hiện:

1. Thêm mô hình **XGBoost**
2. **Cross-Validation** để đánh giá ổn định
3. **RandomizedSearchCV** để tìm hyperparameter tối ưu cho Random Forest và XGBoost
4. So sánh kết quả trước và sau tuning


<!-- @format -->

## 1. Setup & Load Data


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import (
    X_TRAIN_FILE, X_TEST_FILE, Y_TRAIN_FILE, Y_TEST_FILE,
    TARGET_COL, RANDOM_STATE, MODELS_DIR
)
from src.utils import save_model

sns.set_theme(style="whitegrid")

# Load dữ liệu
X_train = pd.read_csv(X_TRAIN_FILE)
X_test = pd.read_csv(X_TEST_FILE)
y_train = pd.read_csv(Y_TRAIN_FILE)[TARGET_COL]
y_test = pd.read_csv(Y_TEST_FILE)[TARGET_COL]

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

<!-- @format -->

## 2. Cross-Validation cho các mô hình

Cross-validation giúp đánh giá mô hình ổn định hơn bằng cách chia dữ liệu train thành nhiều fold.


In [ ]:
def run_cv(name, model, X, y, cv=5):
    """Chạy cross-validation và trả về kết quả trung bình."""
    pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
    scoring = {
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2",
    }
    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    result = {
        "model": name,
        "CV_MAE": round(-scores["test_mae"].mean(), 2),
        "CV_RMSE": round(-scores["test_rmse"].mean(), 2),
        "CV_R2": round(scores["test_r2"].mean(), 4),
    }
    print(f"{name}: MAE={result['CV_MAE']}, RMSE={result['CV_RMSE']}, R²={result['CV_R2']}")
    return result

# Cross-validation cho các model
cv_results = []
cv_results.append(run_cv("Random Forest", RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1), X_train, y_train))
cv_results.append(run_cv("XGBoost", XGBRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1, verbosity=0), X_train, y_train))

cv_df = pd.DataFrame(cv_results).sort_values("CV_R2", ascending=False)
display(cv_df)

<!-- @format -->

## 3. Tuning Random Forest với RandomizedSearchCV

Tìm kiếm bộ hyperparameter tốt nhất cho Random Forest.


In [ ]:
# Tham số tìm kiếm cho Random Forest
rf_param_dist = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [10, 15, 20, 25, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", 0.5],
}

rf_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
])

rf_search = RandomizedSearchCV(
    rf_pipe,
    rf_param_dist,
    n_iter=20,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

rf_search.fit(X_train, y_train)
print(f"\nBest RF params: {rf_search.best_params_}")
print(f"Best CV RMSE: {-rf_search.best_score_:.2f}")

<!-- @format -->

## 4. Tuning XGBoost với RandomizedSearchCV


In [ ]:
# Tham số tìm kiếm cho XGBoost
xgb_param_dist = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [3, 5, 7, 10],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__min_child_weight": [1, 3, 5],
}

xgb_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)),
])

xgb_search = RandomizedSearchCV(
    xgb_pipe,
    xgb_param_dist,
    n_iter=20,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

xgb_search.fit(X_train, y_train)
print(f"\nBest XGBoost params: {xgb_search.best_params_}")
print(f"Best CV RMSE: {-xgb_search.best_score_:.2f}")

<!-- @format -->

## 5. So sánh Baseline vs Tuned trên Test Set


In [ ]:
def eval_on_test(name, model, X_te, y_te):
    """Đánh giá model trên test set."""
    y_pred = model.predict(X_te)
    mae = mean_absolute_error(y_te, y_pred)
    rmse = mean_squared_error(y_te, y_pred) ** 0.5
    r2 = r2_score(y_te, y_pred)
    return {"model": name, "MAE": round(mae, 2), "RMSE": round(rmse, 2), "R2": round(r2, 4)}

# Đánh giá baseline (chưa tuning)
from src.utils import load_model
baseline_rf = load_model(MODELS_DIR / "baseline" / "random_forest.joblib")
baseline_lr = load_model(MODELS_DIR / "baseline" / "linear_regression.joblib")

comparison = []
comparison.append(eval_on_test("Linear Regression", baseline_lr, X_test, y_test))
comparison.append(eval_on_test("Random Forest (baseline)", baseline_rf, X_test, y_test))
comparison.append(eval_on_test("Random Forest (tuned)", rf_search.best_estimator_, X_test, y_test))
comparison.append(eval_on_test("XGBoost (tuned)", xgb_search.best_estimator_, X_test, y_test))

comparison_df = pd.DataFrame(comparison).sort_values("R2", ascending=False)
display(comparison_df)

In [ ]:
# Biểu đồ so sánh R²
fig, ax = plt.subplots(figsize=(10, 5))
df_plot = comparison_df.sort_values("R2")
colors = sns.color_palette("viridis", len(df_plot))
ax.barh(df_plot["model"], df_plot["R2"], color=colors)
for i, v in enumerate(df_plot["R2"]):
    ax.text(v + 0.005, i, f"{v:.4f}", va="center", fontsize=11)
ax.set_xlabel("R² Score")
ax.set_title("So sánh R² - Baseline vs Tuned", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

<!-- @format -->

## 6. Lưu mô hình đã tuning


In [ ]:
# Lưu model đã tuning
save_model(rf_search.best_estimator_, MODELS_DIR / "tuned" / "random_forest_tuned.joblib")
save_model(xgb_search.best_estimator_, MODELS_DIR / "tuned" / "xgboost_tuned.joblib")

# Lưu bảng so sánh
comparison_df.to_csv(MODELS_DIR / "tuned" / "tuning_comparison.csv", index=False)

print("Đã lưu mô hình tuned và bảng so sánh!")
print(f"Best RF params: {rf_search.best_params_}")
print(f"Best XGB params: {xgb_search.best_params_}")

<!-- @format -->

## Nhận xét

- **Hyperparameter tuning** giúp cải thiện hiệu suất so với baseline mặc định.
- **XGBoost** và **Random Forest (tuned)** thường cho kết quả tốt nhất cho bài toán dự đoán giá vé.
- Mô hình tốt nhất sẽ được đánh giá chi tiết ở **Notebook 05**.
